# F1AeroNet — Metric Verification (`final_run_(cp_spcfc)`)

Evaluates `runs/final_run_(cp_spcfc)/best.pt` on the **full 45-sample test split**
and compares against the README baseline:

| Metric | README value |
|--------|-------------|
| Cp MAE | 0.12142 |
| Cp RMSE | 0.63755 |
| Cp R² | 0.0846 |
| WSS MAE | 0.14923 |
| WSS RMSE | 0.79070 |
| WSS AngErr | 17.12° |

**Cells in order:**
1. GPU / environment check
2. Install dependencies
3. Clone code from GitHub
4. Copy dataset from Kaggle input
5. Inject Cd normalisation stats
6. Locate checkpoint
7. Load model
8. Run evaluation on test split
9. Compare against README baseline

**Required Kaggle inputs:**
- `aumrawal29/drivaernet-200` dataset (same as training)
- The `final_run_(cp_spcfc)/best.pt` checkpoint — either committed to the GitHub
  repo (auto-cloned in cell 3) **or** uploaded as a separate Kaggle dataset
  and attached as input (cell 6 searches both locations).

In [ ]:
# ── Cell 1 · GPU / Environment Check ────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("No GPU detected — inference will run on CPU.")

print(f"PyTorch: {torch.__version__}")

TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = f"cu{torch.version.cuda.replace('.', '')}" if torch.cuda.is_available() else "cpu"
print(f"\nInstall tag: torch-{TORCH_VER}+{CUDA_TAG}")

In [ ]:
# ── Cell 2 · Install Dependencies ───────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('torch-geometric')

pyg_url = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'
try:
    pip('torch-scatter', 'torch-sparse', '-f', pyg_url)
    print('torch-scatter / torch-sparse installed.')
except Exception as e:
    print(f'[WARN] C extensions unavailable ({e}). Using pure-Python fallback.')

pip('pyvista', 'vtk', 'scipy', 'pyyaml', 'tqdm')
print('\nAll dependencies installed.')

In [ ]:
# ── Cell 3 · Clone Code from GitHub ─────────────────────────────────────────
import os, subprocess, sys

REPO_URL = 'https://github.com/aumrawal/GDL_Cars.git'
REPO_DIR = '/kaggle/working/GDL_Cars'

if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Clone complete.')
else:
    print(f'{REPO_DIR} already exists — pulling latest.')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('\nRepo contents:')
for f in sorted(os.listdir(REPO_DIR)):
    print(f'  {f}')

In [ ]:
# ── Cell 4 · Copy Dataset ────────────────────────────────────────────────────
import shutil, os

SRC_DATASET = '/kaggle/input/drivaernet-200'
DST_DATASET = '/kaggle/working/drivaernet_real'

if not os.path.exists(DST_DATASET):
    print(f'Copying dataset...\n  {SRC_DATASET}\n  → {DST_DATASET}')
    shutil.copytree(SRC_DATASET, DST_DATASET)
    print('Copy complete.')
else:
    print(f'{DST_DATASET} already exists, skipping copy.')

mesh_dir = os.path.join(DST_DATASET, 'meshes')
if os.path.isdir(mesh_dir):
    vtps = [f for f in os.listdir(mesh_dir) if f.endswith('.vtp')]
    print(f'\nFound {len(vtps)} VTP meshes.')
else:
    raise RuntimeError(f'No meshes/ directory found in {DST_DATASET}.')

In [ ]:
# ── Cell 5 · Inject Cd Normalisation Stats ───────────────────────────────────
# Stats computed from the 210-sample training split during the original run.
# Required by DrivAerNetDataset to z-score y_cd targets on the test split.
import json, os

DATA_ROOT  = '/kaggle/working/drivaernet_real'
STATS_PATH = os.path.join(DATA_ROOT, 'cd_stats.json')

if not os.path.exists(STATS_PATH):
    s = {'cd_mean': 0.241828, 'cd_std': 0.017059, 'n_samples': 210}
    with open(STATS_PATH, 'w') as f:
        json.dump(s, f, indent=2)
    print(f'Wrote cd_stats.json: mean={s["cd_mean"]}  std={s["cd_std"]}')
else:
    with open(STATS_PATH) as f:
        s = json.load(f)
    print(f'cd_stats.json already exists: mean={s["cd_mean"]}  std={s["cd_std"]}')

In [ ]:
# ── Cell 6 · Locate Checkpoint ──────────────────────────────────────────────
# Search order:
#   1. Cloned repo  (if runs/final_run_(cp_spcfc)/best.pt was committed)
#   2. Kaggle input (if uploaded as a dataset, e.g. /kaggle/input/<dataset>/best.pt)
import os

CKPT_CANDIDATES = [
    os.path.join(REPO_DIR, 'runs', 'final_run_(cp_spcfc)', 'best.pt'),
    # If you uploaded the checkpoint as a Kaggle dataset, add its path here:
    # '/kaggle/input/<your-dataset-slug>/best.pt',
]

# Also scan any Kaggle input directory that contains a best.pt
kaggle_input = '/kaggle/input'
if os.path.isdir(kaggle_input):
    for entry in os.listdir(kaggle_input):
        candidate = os.path.join(kaggle_input, entry, 'best.pt')
        if os.path.exists(candidate) and candidate not in CKPT_CANDIDATES:
            CKPT_CANDIDATES.append(candidate)

CKPT_PATH = None
for p in CKPT_CANDIDATES:
    if os.path.exists(p):
        CKPT_PATH = p
        print(f'Found checkpoint: {CKPT_PATH}')
        print(f'Size: {os.path.getsize(CKPT_PATH) / 1024:.1f} KB')
        break

if CKPT_PATH is None:
    print('[ERROR] Checkpoint not found. Options:')
    print('  1. Commit runs/final_run_(cp_spcfc)/best.pt to GitHub and re-run cell 3.')
    print('  2. Upload best.pt as a Kaggle dataset and add its path to CKPT_CANDIDATES above.')
    raise FileNotFoundError('final_run_(cp_spcfc)/best.pt not found in any search location.')

In [ ]:
# ── Cell 7 · Load Model ──────────────────────────────────────────────────────
import os, sys, torch

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from models.f1_net import F1AeroNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
print(f'Epoch : {ckpt.get("epoch")}  |  best_val_loss = {ckpt.get("best_val", float("nan")):.6f}')

# Read model config embedded in the checkpoint
saved_cfg  = ckpt.get('cfg', {})
model_cfg  = saved_cfg.get('model', {
    'in_channels': 9,
    'layer_types': [[16, 2], [16, 2], [16, 2], [24, 1], [16, 1]],
    'nonlin_samples': 5,
    'head_dropout': 0.1,
    'cd_head_dropout': 0.5,
    'break_symmetry_final': True,
})
print('\nModel config:', model_cfg)

model = F1AeroNet.from_config(model_cfg).to(device)
model.load_state_dict(ckpt['model'])
model.eval()

params = model.count_parameters()
print(f'\nParameters: {params["total"]:,} total')

In [ ]:
# ── Cell 8 · Run Evaluation on Full Test Split ───────────────────────────────
import os, sys
from torch_geometric.loader import DataLoader

from data.drivaernet_dataset import DrivAerNetDataset
from eval.evaluator import evaluate

test_ds = DrivAerNetDataset(
    data_root    = DATA_ROOT,
    split        = 'test',
    max_vertices = 50000,
)
print(f'Test split: {len(test_ds)} samples')

if len(test_ds) == 0:
    raise RuntimeError('No test samples found. Check DATA_ROOT and split.json.')

test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

# OOM guard: Kaggle T4 has 16 GB; skip any mesh with >150k edges
# (evaluate() in eval/evaluator.py does not OOM-guard — wrap the loader)
import torch

metrics = evaluate(model, test_loader, device, verbose=True)

In [ ]:
# ── Cell 9 · Compare Against README Baseline ────────────────────────────────
README = {
    'cp_mae':        0.12142,
    'cp_rmse':       0.63755,
    'cp_r2':         0.0846,
    'wss_mae':       0.14923,
    'wss_rmse':      0.79070,
    'wss_angle_err': 17.12,
}

TOL = 0.02  # 2 % relative tolerance → PASS

ROWS = [
    ('Cp  MAE',      'cp_mae'),
    ('Cp  RMSE',     'cp_rmse'),
    ('Cp  R²',       'cp_r2'),
    ('WSS MAE',      'wss_mae'),
    ('WSS RMSE',     'wss_rmse'),
    ('WSS AngErr',   'wss_angle_err'),
]

print()
print(f"{'─'*64}")
print(f"  {'Metric':<16}  {'Computed':>10}  {'README':>10}  {'Δ':>8}  Pass?")
print(f"{'─'*64}")

all_pass = True
for label, key in ROWS:
    val = metrics.get(key)
    ref = README.get(key)
    if val is None:
        print(f"  {label:<16}  {'N/A':>10}  {ref:>10.5f}  {'—':>8}  skipped")
        continue
    delta = val - ref
    rel   = abs(delta) / (abs(ref) + 1e-10)
    mark  = 'PASS' if rel <= TOL else 'FAIL'
    if mark == 'FAIL':
        all_pass = False
    unit  = '°' if key == 'wss_angle_err' else ''
    print(f"  {label:<16}  {val:>10.5f}  {ref:>10.5f}  {delta:>+8.4f}  {mark}")

print(f"{'─'*64}")
print()
if all_pass:
    print('  All metrics within 2% of README baseline. Verification PASSED.')
else:
    print('  One or more metrics deviate >2% from README baseline.')
    print('  This may indicate a checkpoint/dataset mismatch — check CKPT_PATH.')
print()